# Irrigation Prediction — Pandas and NumPy Data Analysis

**Use case:** Explore field, crop and weather factors associated with `Irrigation_Need` (`Low`, `Medium`, `High`).  
**Dataset:** `irrigation_prediction.csv` (10,000 records, 20 original columns).  

`Irrigation_Need` is the label, so it should be excluded from input features in any later prediction model. Associations here are descriptive; they do not prove causes or model accuracy.


## 1. Read the dataset

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
csv_path = Path('irrigation_prediction.csv')
df = pd.read_csv(csv_path)
moisture_array = np.asarray(df['Soil_Moisture'])
print('Dataset shape:', df.shape)
print('NumPy soil-moisture array:', moisture_array.shape, moisture_array.dtype)
print(df.head(5).to_string(index=False))


Dataset shape: (10000, 20)
NumPy soil-moisture array: (10000,) float64
Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  Sunlight_Hours  Wind_Speed_kmh Crop_Type Crop_Growth_Stage Season Irrigation_Type Water_Source  Field_Area_hectare Mulching_Used  Previous_Irrigation_mm  Region Irrigation_Need
     Clay     6.14          36.48            0.42                     2.17          21.90     31.19      1167.70            4.01            1.97     Wheat        Vegetative   Rabi         Rainfed    Reservoir                4.73           Yes                    1.98   South             Low
     Silt     6.41          50.56            0.38                     0.23          36.50     26.01       831.28           10.72           16.82     Maize         Flowering   Zaid           Canal  Groundwater               12.22           Yes                   33.56 Central          Medium
    Sandy     7.71          40.07            1.09       

## 2. Detailed analysis: `info()` and `describe()`

In [2]:
df.info()
print('\nNumeric summary:')
print(df.describe().round(2).to_string())
print('\nCategorical summary:')
print(df.describe(include='object').to_string())
print('Inference: Each row is one field observation; Irrigation_Need is a three-class categorical target.')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Soil_Type                10000 non-null  object 
 1   Soil_pH                  10000 non-null  float64
 2   Soil_Moisture            10000 non-null  float64
 3   Organic_Carbon           10000 non-null  float64
 4   Electrical_Conductivity  10000 non-null  float64
 5   Temperature_C            10000 non-null  float64
 6   Humidity                 10000 non-null  float64
 7   Rainfall_mm              10000 non-null  float64
 8   Sunlight_Hours           10000 non-null  float64
 9   Wind_Speed_kmh           10000 non-null  float64
 10  Crop_Type                10000 non-null  object 
 11  Crop_Growth_Stage        10000 non-null  object 
 12  Season                   10000 non-null  object 
 13  Irrigation_Type          10000 non-null  object 
 14  Water_Source           

## 3. Check null values

In [3]:
null_counts = df.isna().sum()
print(null_counts.to_string())
print('Total missing cells:', int(null_counts.sum()))


Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
Total missing cells: 0


## 4. Handle missing values with `fillna(..., inplace=True)`

In [4]:
# The supplied file has no missing cells. Work on a copy so this demonstration
# does not invent missing data in the analysis frame or alter the source CSV.
imputation_demo = df[['Soil_Moisture', 'Soil_Type']].head(5).copy()
imputation_demo.loc[imputation_demo.index[0], 'Soil_Moisture'] = np.nan
imputation_demo.loc[imputation_demo.index[1], 'Soil_Type'] = np.nan
numeric_fill = df['Soil_Moisture'].median()
category_fill = df['Soil_Type'].mode().iloc[0]
print('Before:', imputation_demo.isna().sum().to_dict())
imputation_demo.fillna({'Soil_Moisture': numeric_fill, 'Soil_Type': category_fill}, inplace=True)
print('After:', imputation_demo.isna().sum().to_dict())
print('Filled with median soil moisture', numeric_fill, 'and modal soil type', category_fill)
print('Inference: A future dataset can use median/mode for predictors; target labels should not be imputed.')


Before: {'Soil_Moisture': 1, 'Soil_Type': 1}
After: {'Soil_Moisture': 0, 'Soil_Type': 0}
Filled with median soil moisture 37.239999999999995 and modal soil type Sandy
Inference: A future dataset can use median/mode for predictors; target labels should not be imputed.


## 5. Sorting with conditional filtering

In [5]:
top_driest = df.sort_values('Soil_Moisture').head(8)
hot_dry = df.loc[(df['Temperature_C'] > 30) & (df['Soil_Moisture'] < 25)]
top_hot_dry = hot_dry.sort_values('Temperature_C', ascending=False).head(5)
print('Eight driest observations:')
print(top_driest[['Soil_Moisture', 'Temperature_C', 'Irrigation_Need']].to_string(index=False))
print('\nFive hottest observations among hot and dry fields:')
print(top_hot_dry[['Soil_Moisture', 'Temperature_C', 'Irrigation_Need']].to_string(index=False))
print('Hot and dry matching rows:', len(hot_dry))


Eight driest observations:
 Soil_Moisture  Temperature_C Irrigation_Need
          8.00          33.49          Medium
          8.01          40.38            High
          8.02          39.31          Medium
          8.02          14.05             Low
          8.02          24.94          Medium
          8.03          40.82          Medium
          8.03          21.27          Medium
          8.03          39.20          Medium

Five hottest observations among hot and dry fields:
 Soil_Moisture  Temperature_C Irrigation_Need
         23.31          42.00          Medium
         16.97          41.97          Medium
         20.05          41.97             Low
         10.44          41.96          Medium
         21.78          41.94            High
Hot and dry matching rows: 1175


## 6. Frequency listings: target and soil type

In [6]:
need_freq = df['Irrigation_Need'].value_counts()
soil_freq = df['Soil_Type'].value_counts()
print('Irrigation need counts and percentages:')
print(pd.DataFrame({'count': need_freq, 'percent': (need_freq / len(df) * 100).round(2)}).to_string())
print('\nSoil type counts:')
print(soil_freq.to_string())
print('Inference: The High class is uncommon, so a future classifier needs class-wise evaluation.')


Irrigation need counts and percentages:
                 count  percent
Irrigation_Need                
Low               5864    58.64
Medium            3800    38.00
High               336     3.36

Soil type counts:
Soil_Type
Sandy    2536
Silt     2501
Loamy    2486
Clay     2477
Inference: The High class is uncommon, so a future classifier needs class-wise evaluation.


## 7. Sort rows and columns; explicit and implicit indexing

In [7]:
# CSV has no unique field identifier. Create a stable notebook-only row ID.
indexed = df.set_axis(pd.Index(np.arange(1, len(df) + 1), name='Observation_ID'))
print('Explicit .loc[1]:')
print(indexed.loc[1, ['Soil_Type', 'Soil_Moisture', 'Irrigation_Need']].to_string())
print('\nImplicit .iloc[5]:')
print(indexed.iloc[5][['Soil_Type', 'Soil_Moisture', 'Irrigation_Need']].to_string())
sorted_rows = indexed.sort_index(ascending=False)
sorted_columns = indexed.sort_index(axis=1)
print('\nFirst descending row ID:', sorted_rows.index[0])
print('Alphabetically sorted columns:', sorted_columns.columns.tolist())


Explicit .loc[1]:
Soil_Type           Clay
Soil_Moisture      36.48
Irrigation_Need      Low

Implicit .iloc[5]:
Soil_Type            Silt
Soil_Moisture        20.5
Irrigation_Need    Medium

First descending row ID: 10000
Alphabetically sorted columns: ['Crop_Growth_Stage', 'Crop_Type', 'Electrical_Conductivity', 'Field_Area_hectare', 'Humidity', 'Irrigation_Need', 'Irrigation_Type', 'Mulching_Used', 'Organic_Carbon', 'Previous_Irrigation_mm', 'Rainfall_mm', 'Region', 'Season', 'Soil_Moisture', 'Soil_Type', 'Soil_pH', 'Sunlight_Hours', 'Temperature_C', 'Water_Source', 'Wind_Speed_kmh']


## 8. Access selected columns under compound conditions

In [8]:
case1 = df.loc[(df['Soil_Moisture'] < 25) & (df['Temperature_C'] > 30),
               ['Soil_Moisture', 'Temperature_C', 'Irrigation_Need']]
case2 = df.loc[(df['Rainfall_mm'] < 500) | (df['Humidity'] < 40),
               ['Rainfall_mm', 'Humidity', 'Irrigation_Need']]
case3 = df.loc[(df['Crop_Growth_Stage'] == 'Flowering') &
               (df['Season'].isin(['Kharif', 'Zaid'])),
               ['Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Need']]
for label, subset in [('Hot AND dry', case1), ('Low rain OR low humidity', case2),
                      ('Flowering in Kharif/Zaid', case3)]:
    print('\n', label, '| matching rows:', len(subset))
    print(subset.head(5).to_string(index=False))



 Hot AND dry | matching rows: 1175
 Soil_Moisture  Temperature_C Irrigation_Need
         12.75          37.22          Medium
         20.50          33.34          Medium
         19.58          31.14          Medium
         11.85          35.21            High
         17.77          34.84          Medium

 Low rain OR low humidity | matching rows: 3682
 Rainfall_mm  Humidity Irrigation_Need
     1167.70     31.19             Low
      831.28     26.01          Medium
      306.26     43.32          Medium
      402.92     62.51          Medium
      443.45     87.34             Low

 Flowering in Kharif/Zaid | matching rows: 1614
Crop_Type Crop_Growth_Stage Season Irrigation_Need
    Maize         Flowering   Zaid          Medium
     Rice         Flowering Kharif          Medium
    Maize         Flowering Kharif             Low
     Rice         Flowering Kharif            High
Sugarcane         Flowering Kharif          Medium


## 9. Minimum and maximum values

In [9]:
measurements = ['Soil_Moisture', 'Temperature_C', 'Humidity', 'Rainfall_mm',
                'Previous_Irrigation_mm']
print(pd.DataFrame({'minimum': df[measurements].min(),
                    'maximum': df[measurements].max()}).round(2).to_string())
print('\nDriest observation:')
print(df.loc[df['Soil_Moisture'].idxmin(),
             ['Soil_Type', 'Crop_Type', 'Soil_Moisture', 'Irrigation_Need']].to_string())
print('\nWettest observation:')
print(df.loc[df['Soil_Moisture'].idxmax(),
             ['Soil_Type', 'Crop_Type', 'Soil_Moisture', 'Irrigation_Need']].to_string())


                        minimum  maximum
Soil_Moisture              8.00    65.00
Temperature_C             12.00    42.00
Humidity                  25.00    95.00
Rainfall_mm                0.38  2499.69
Previous_Irrigation_mm     0.02   119.99

Driest observation:
Soil_Type            Clay
Crop_Type          Potato
Soil_Moisture         8.0
Irrigation_Need    Medium

Wettest observation:
Soil_Type               Clay
Crop_Type          Sugarcane
Soil_Moisture           65.0
Irrigation_Need          Low


## 10. GroupBy on one and multiple columns

In [10]:
by_need = df.groupby('Irrigation_Need')['Soil_Moisture'].mean().reindex(['Low', 'Medium', 'High'])
by_soil_need = df.groupby(['Soil_Type', 'Irrigation_Need'])['Soil_Moisture'].mean()
print('Average soil moisture by irrigation need:')
print(by_need.round(2).to_string())
print('\nAverage soil moisture by soil and need:')
print(by_soil_need.round(2).to_string())
print('Inference: Compare within soil types because soil composition may affect moisture readings.')


Average soil moisture by irrigation need:
Irrigation_Need
Low       41.72
Medium    31.16
High      19.71

Average soil moisture by soil and need:
Soil_Type  Irrigation_Need
Clay       High               17.66
           Low                42.06
           Medium             31.27
Loamy      High               18.93
           Low                41.45
           Medium             31.02
Sandy      High               21.19
           Low                41.70
           Medium             30.96
Silt       High               20.80
           Low                41.70
           Medium             31.38
Inference: Compare within soil types because soil composition may affect moisture readings.


## 11. Add a derived column

In [11]:
# Multiplying irrigation depth (mm) by hectares converts to cubic metres:
# 1 mm over 1 hectare = 10 m^3. This is past irrigation, not a recommended dose.
df['Previous_Irrigation_Volume_m3'] = (
    df['Previous_Irrigation_mm'] * df['Field_Area_hectare'] * 10
)
print(df[['Previous_Irrigation_mm', 'Field_Area_hectare',
          'Previous_Irrigation_Volume_m3']].head().round(2).to_string(index=False))


 Previous_Irrigation_mm  Field_Area_hectare  Previous_Irrigation_Volume_m3
                   1.98                4.73                          93.65
                  33.56               12.22                        4101.03
                  34.62                5.52                        1911.02
                  84.03                1.43                        1201.63
                  60.86                2.52                        1533.67


## 12. Aggregate functions with GroupBy

In [12]:
agg1 = df.groupby('Irrigation_Need')['Soil_Moisture'].agg(['count', 'mean', 'min', 'max', 'std'])
agg2 = df.groupby(['Season', 'Irrigation_Need']).agg(
    average_moisture=('Soil_Moisture', 'mean'),
    average_temperature=('Temperature_C', 'mean'),
    average_rainfall=('Rainfall_mm', 'mean'),
    total_fields=('Irrigation_Need', 'size'))
print('Soil moisture by target:')
print(agg1.round(2).to_string())
print('\nWeather and field counts by season and target:')
print(agg2.round(2).to_string())


Soil moisture by target:
                 count   mean   min    max    std
Irrigation_Need                                  
High               336  19.71  8.01  64.22  10.84
Low               5864  41.72  8.02  65.00  14.44
Medium            3800  31.16  8.00  64.99  16.83

Weather and field counts by season and target:
                        average_moisture  average_temperature  average_rainfall  total_fields
Season Irrigation_Need                                                                       
Kharif High                        18.76                33.05            748.45           113
       Low                         42.01                25.52           1327.79          1947
       Medium                      30.40                28.69           1162.28          1302
Rabi   High                        18.78                32.77            652.85           115
       Low                         41.75                25.58           1354.91          2016
       Medium      

## 13. Select a group by name or condition

In [13]:
high_group = df.groupby('Irrigation_Need').get_group('High')
dry_soil_groups = df.groupby('Soil_Type').filter(lambda group: group['Soil_Moisture'].mean() < 37)
print('High need rows:', len(high_group))
print(high_group[['Soil_Type', 'Soil_Moisture', 'Temperature_C']].head().to_string(index=False))
print('Soil types whose mean moisture is below 37%:', sorted(dry_soil_groups['Soil_Type'].unique()))
print('Rows in those soil types:', len(dry_soil_groups))


High need rows: 336
Soil_Type  Soil_Moisture  Temperature_C
     Clay          11.85          35.21
    Loamy           9.60          20.66
     Silt          10.46          35.23
    Loamy          11.29          41.82
     Clay          20.35          39.64
Soil types whose mean moisture is below 37%: ['Loamy', 'Sandy']
Rows in those soil types: 5022


## 14. Correlation between numeric columns

In [14]:
numeric_cols = ['Soil_Moisture', 'Temperature_C', 'Humidity',
                'Rainfall_mm', 'Previous_Irrigation_mm']
print(df[numeric_cols].corr().round(3).to_string())
r = df['Soil_Moisture'].corr(df['Rainfall_mm'])
print('\nPearson correlation, soil moisture vs rainfall:', round(r, 3))
print('Inference: Correlation is descriptive and does not establish causation or predictive performance.')


                        Soil_Moisture  Temperature_C  Humidity  Rainfall_mm  Previous_Irrigation_mm
Soil_Moisture                   1.000         -0.005     0.007       -0.006                  -0.009
Temperature_C                  -0.005          1.000    -0.009        0.008                   0.004
Humidity                        0.007         -0.009     1.000       -0.007                  -0.002
Rainfall_mm                    -0.006          0.008    -0.007        1.000                   0.001
Previous_Irrigation_mm         -0.009          0.004    -0.002        0.001                   1.000

Pearson correlation, soil moisture vs rainfall: -0.006
Inference: Correlation is descriptive and does not establish causation or predictive performance.


## 15. Transformation: normalization

In [15]:
# Descriptive scaling on the full dataset. For a prediction model, fit scaling
# parameters on the training split only to avoid data leakage.
moisture_min, moisture_max = df['Soil_Moisture'].min(), df['Soil_Moisture'].max()
df['Soil_Moisture_minmax'] = (df['Soil_Moisture'] - moisture_min) / (moisture_max - moisture_min)
df['Soil_Moisture_zscore'] = (df['Soil_Moisture'] - df['Soil_Moisture'].mean()) / df['Soil_Moisture'].std()
print(df[['Soil_Moisture', 'Soil_Moisture_minmax', 'Soil_Moisture_zscore']].head().round(3).to_string(index=False))
print('Min-max range:', (df['Soil_Moisture_minmax'].min(), df['Soil_Moisture_minmax'].max()))


 Soil_Moisture  Soil_Moisture_minmax  Soil_Moisture_zscore
         36.48                 0.500                -0.030
         50.56                 0.747                 0.827
         40.07                 0.563                 0.189
         12.75                 0.083                -1.474
         18.58                 0.186                -1.119
Min-max range: (np.float64(0.0), np.float64(1.0))


## 16. Joining, merging and concatenation

In [16]:
# Lookup is a descriptive severity rank, not a predicted label.
need_lookup = pd.DataFrame({'Irrigation_Need': ['Low', 'Medium', 'High'],
                            'Need_Rank': [1, 2, 3]})
merged = df.merge(need_lookup, on='Irrigation_Need', how='left', validate='many_to_one')
joined = df.join(need_lookup.set_index('Irrigation_Need'), on='Irrigation_Need', validate='many_to_one')
split_at = len(df) // 2
concatenated = pd.concat([df.iloc[:split_at], df.iloc[split_at:]], axis=0)
print('Merged rows:', len(merged), '| missing ranks:', merged['Need_Rank'].isna().sum())
print('Joined rows:', len(joined), '| same rank as merge:', joined['Need_Rank'].reset_index(drop=True).equals(merged['Need_Rank']))
print('Concatenated rows:', len(concatenated), '| exactly reconstructs frame:', concatenated.equals(df))
print(merged[['Irrigation_Need', 'Need_Rank']].drop_duplicates().sort_values('Need_Rank').to_string(index=False))


Merged rows: 10000 | missing ranks: 0
Joined rows: 10000 | same rank as merge: True
Concatenated rows: 10000 | exactly reconstructs frame: True
Irrigation_Need  Need_Rank
            Low          1
         Medium          2
           High          3


## Use-case conclusion

The supplied file supports exploratory analysis of irrigation need. The class distribution is imbalanced, with relatively few `High` observations. Soil moisture, weather, crop, and field variables can be candidate predictors, but this notebook does not train a model or claim forecast accuracy. A model experiment should use a stratified train/test split, fit any imputer or scaler only on training data, and report a confusion matrix plus class-wise precision and recall.
